# Component 1 — Observational Intersectional Fairness (Synthetic Example)

This notebook:
1) Generates synthetic clinical-style data with intersectional groups.
2) Splits dev/test by state.
3) Runs Component 1 fairness evaluation with LightGBM.

In [ ]:
import numpy as np
import pandas as pd

!pip install --force-reinstall --no-deps --no-cache-dir \
    git+https://github.com/vsubbian/FairLogue.git

import FairLogue

from sklearn.metrics import roc_auc_score
from lightgbm import LGBMClassifier

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def make_synth_component1(n=8000, seed=42):
    rng = np.random.default_rng(seed)

    #Data example derived from 2 year stroke risk prediction task
    #States for dev/test split
    development_states = ["WI", "MN", "AZ", "MA", "GA", "PA", "TX"]
    test_states = ["FL", "MI", "IL", "AL"]
    all_states = development_states + test_states

    state = rng.choice(all_states, size=n, replace=True)
    state_of_residence_source_value = np.array([f"PII State: {s}" for s in state])

    #Protected attributes
    gender = rng.integers(0, 2, size=n)  # 0=female,1=male
    race = rng.choice(["White", "Black or African American"], size=n, p=[0.7, 0.3])

    #Clinical covariates
    age = rng.normal(60, 12, size=n).clip(18, 90)
    chf = rng.binomial(1, sigmoid((age - 65) / 10), size=n)
    htn = rng.binomial(1, sigmoid((age - 55) / 10), size=n)
    diabetes = rng.binomial(1, 0.2 + 0.1 * (htn == 1), size=n)
    stroke_prior = rng.binomial(1, 0.08 + 0.06 * (chf == 1), size=n)
    age_75 = (age >= 75).astype(int)

    #Encode factors in a way that resembles your pipelines
    gender_factor = gender.astype(int)
    race_factor = (race == "Black or African American").astype(int)  # 1=Black,0=White

    #Outcome with (controlled) intersectional effect
    #Add a mild interaction that Component 1 should detect as disparity.
    inter_effect = 0.35 * ((race_factor == 1) & (gender_factor == 0))  # Black+Female higher risk
    state_effect = 0.15 * np.isin(state, ["PA", "TX"]).astype(float)

    logit = (
        -3.0
        + 0.04 * (age - 50)
        + 0.7 * chf
        + 0.5 * htn
        + 0.6 * diabetes
        + 0.8 * stroke_prior
        + 0.25 * age_75
        + inter_effect
        + state_effect
    )
    p = sigmoid(logit)
    target = rng.binomial(1, p, size=n)

    df = pd.DataFrame({
        "state_of_residence_source_value": state_of_residence_source_value,
        "gender_factor": gender_factor,
        "race_factor": race_factor,
        "age": age,
        "chf": chf,
        "htn": htn,
        "age_75": age_75,
        "diabetes": diabetes,
        "stroke_prior": stroke_prior,
        "target": target,
        # optional human-readable versions
        "race": race,
        "gender": np.where(gender_factor == 1, "Male", "Female"),
    })

    return df, development_states, test_states

df, development_states, test_states = make_synth_component1(n=8000, seed=42)
df.head()

In [ ]:
features = ["age", "gender_factor", "race_factor", "chf", "htn", "age_75", "diabetes", "stroke_prior"]

dev_df = df[df["state_of_residence_source_value"].isin([f"PII State: {s}" for s in development_states])].copy()
test_df = df[df["state_of_residence_source_value"].isin([f"PII State: {s}" for s in test_states])].copy()

print(dev_df.shape, test_df.shape)

In [ ]:
MODEL_PARAMS = {
    "objective": "binary",
    "n_estimators": 600,
    "num_leaves": 31,
    "learning_rate": 0.05,
    "max_depth": -1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "class_weight": "balanced",
}

lgbm = LGBMClassifier(**MODEL_PARAMS)
lgbm.fit(dev_df[features], dev_df["target"])

p_test = lgbm.predict_proba(test_df[features])[:, 1]
print("Test AUROC:", roc_auc_score(test_df["target"], p_test))

In [ ]:
results, figures, intermediates = FairLogue.Component1.evaluate_intersectional_fairness(
    df=df,
    # If your Component 1 supports explicit split override, use these:
    train_df=dev_df,
    test_df=test_df,
    outcome="target",
    protected_1="race_factor",
    protected_2="gender_factor",
    features=features,
    model_type="lgbm",
    model_params=MODEL_PARAMS,
    threshold=0.5,
    make_plots=True,
    return_intermediates=True,
    return_non_intersectional=True,
    min_group_size=50,
    require_class_balance=True,
)

results

In [ ]:
if isinstance(figures, dict):
    for name, fig in figures.items():
        print(name)
        display(fig)